In [73]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
import os

openai_client = OpenAI(
    api_key=os.getenv('GROQ_API_KEY'),
    base_url='https://api.groq.com/openai/v1'
)

In [74]:
def llm(prompt):
    response = openai_client.chat.completions.create(
        model='llama-3.3-70b-versatile',   # or any Groq-supported model
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    return response.choices[0].message.content

In [75]:
question = 'I just discovered the course. Can I join now?'

answer = llm(question)

print(answer)

I'm excited that you're interested in the course. However, I need a bit more information from you. Could you please provide more context or details about the course you're referring to? That way, I can better assist you with your query. What type of course is it, and where is it being offered?


In [76]:
context = '''
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
'''

In [77]:
prompt = f'''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
'''

In [78]:
print(prompt)


Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
I just discovered the course. Can I join now?

Context:

I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students partici

In [79]:
answer = llm(prompt)
print(answer)

Yes, you can join now. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.


RAG retrieval augumentation generation
retrieval search
knowlwdge base => contains the relevant info where we search 
Augumneted generation=> where we send the relevent data to llm 
3 step=> send, build prompt, send data to llm

In [80]:
import requests
docs_url= 'https://datatalks.club/faq/json/courses.json'
response=requests.get(docs_url)
course_raw=response.json()

In [81]:
course_raw

[{'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 472},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 79},
 {'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 402},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 255}]

In [82]:
# This returns a list of courses. Each course has a path field that points to its FAQ data. Let's fetch all the FAQ documents from all courses:

documents = []
url_prefix = 'https://datatalks.club/faq'

for course in course_raw:
    course_url = f'{url_prefix}{course['path']}'

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)


1208

In [83]:
documents[2]

{'id': '4d5aa45b03',
 'course': 'machine-learning-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Are Jupyter Notebooks used?',
 'answer': 'Yes. You’ll work extensively with notebooks alongside standard Python files and CLI tools.'}

sometime needs to parse data from website to prepare data
Library allow to store and search from documents
Index data such a way that get the most similar and relevant data
search library=> elastic search lucin, solr
light weight elastic search=> text fiels keyword field
text field are fields used to perform search like section question and answer
keyword fields exact match form, select from index where course='data-engineering'=> 
restrict search state to particular subset



In [84]:
from minsearch import Index

index=Index(
    text_fields=['question','section','answer'],
    keyword_fields=['course']
)

index.fit(documents)

In [85]:
search_result= index.search(
    question, filter_dict={'course':'llm_zoomcamp'},
    num_results=5
)

In [86]:
def search(question, course='llm-zoomcamp'):
    boost_dict = {'question': 2.0, 'section': 0.5}
    filter_dict = {'course': course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

Boosting when we search results we can boost result,  
one field question is more import then answer field for example
certificate in question field more important in certificate in answer field
by default importance one , if <1 less important  -->

In [87]:
question = 'I just discovered the course. Can I join now?'


In [88]:
search_result= search(question)
search_result

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

Build prompt
when we build ai prompt, it has 2 part
1st part: never change, instruction
2nd: user: user prompt, it changes

In [89]:
instruction='''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know.
'''


In [90]:
user_prompt_template = '''
Question:
{question}

Context:
{context}
'''

function takes questions, source results and convert to prompt template 

In [91]:
search_result

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

turn dict to llm readable format, easy for llm 
turn dict to string 

In [92]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc['section'])
        lines.append('Q: ' + doc['question'])
        lines.append('A: ' + doc['answer'])
        lines.append('')

    return '\n'.join(lines).strip()

In [93]:
context= build_context(search_result)
print(context)


General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.

You can only peer-review projects at the time the course is run

In [94]:
def build_prompt(question,search_result):
    context=build_context(search_result)
    prompt= user_prompt_template.format(
        question=question, 
        context=context
    )
    return prompt.strip()

In [95]:
prompt= build_prompt(question, search_result)
print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

Add LLM
llm can figure out what the possible answer even without instruction


In [96]:
response = openai_client.chat.completions.create(
    model='llama-3.3-70b-versatile',   # or any Groq-supported model
    messages=[
        {"role": "user", "content": prompt}
    ]
)

In [99]:
response.choices[0].message.content

'Yes, you can still join the course. However, if you want to receive a certificate, you will need to submit your project while submissions are still being accepted. Keep in mind that to get a certificate, you must finish the course with a "live" cohort, as certificates are not awarded for self-paced mode.'

In [100]:
print(response.model_dump_json(indent=2))

{
  "id": "chatcmpl-636ebd56-b39f-4b99-ba03-cfe0f2c3cdcd",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "Yes, you can still join the course. However, if you want to receive a certificate, you will need to submit your project while submissions are still being accepted. Keep in mind that to get a certificate, you must finish the course with a \"live\" cohort, as certificates are not awarded for self-paced mode.",
        "refusal": null,
        "role": "assistant",
        "annotations": null,
        "audio": null,
        "function_call": null,
        "tool_calls": null
      }
    }
  ],
  "created": 1778927573,
  "model": "llama-3.3-70b-versatile",
  "object": "chat.completion",
  "service_tier": "on_demand",
  "system_fingerprint": "fp_dae98b5ecb",
  "usage": {
    "completion_tokens": 65,
    "prompt_tokens": 366,
    "total_tokens": 431,
    "completion_tokens_details": null,
    "prompt_token

usage => tells about no of input tokrn, cache token , output token, response token

In [102]:
response.usage

CompletionUsage(completion_tokens=65, prompt_tokens=366, total_tokens=431, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.055939712, prompt_time=0.018858373, completion_time=0.171984351, total_time=0.190842724)

2 api => response and chat complition

system prompt is hidden, inside chatgpt in history
user prompt 

In [105]:
message_history=[
    {'role': 'developer', 'content': instruction},
    {'role': 'user', 'content': prompt}
]

response = openai_client.chat.completions.create(
    model='llama-3.3-70b-versatile',   # or any Groq-supported model
    messages=message_history
)

In [115]:
def llm(instructions, user_prompt, model='llama-3.3-70b-versatile'):
    message_history = [
        {'role': 'developer', 'content': instructions},
        {'role': 'user', 'content': user_prompt}
    ]

    response = openai_client.chat.completions.create(
        model=model,
        messages=message_history
    )

    return response.choices[0].message.content

In [116]:
def rag(query, model='llama-3.3-70b-versatile'):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(instruction, prompt, model=model)
    return answer

In [117]:
answer = rag('I just discovered the course. Can I join now?')
print(answer)

Yes, you can still join the course. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted. Keep in mind that you must finish the course with a "live" cohort to be eligible for a certificate, as self-paced mode does not qualify for certification.
